# vLLM 배포

- 목적: Hugging Face에 업로드한 merged model을 vLLM OpenAI-compatible server로 배포
- 입력 모델: 2번 노트북에서 업로드한 model repo
- 제외 범위: dataset 로드, 별도 평가 코드
- RunPod 기준: L40S 1장, port 8000

In [ ]:
!cd /workspace && rm -rf vllm-gemma4-fix
!cd /workspace && git clone https://github.com/vllm-project/vllm.git vllm-gemma4-fix
!cd /workspace/vllm-gemma4-fix && git fetch origin pull/43812/head:gemma4-kv-shared-k-norm-sft
!cd /workspace/vllm-gemma4-fix && git checkout gemma4-kv-shared-k-norm-sft

%env VLLM_USE_PRECOMPILED=1
%pip install -e /workspace/vllm-gemma4-fix
%pip install --no-cache-dir -U openai requests

In [ ]:
import torch
import transformers
import vllm

print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("transformers:", transformers.__version__)
print("vllm:", vllm.__version__)
print("vllm path:", vllm.__file__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

path = Path("/workspace/vllm-gemma4-fix/vllm/model_executor/models/gemma4.py")
lines = path.read_text().splitlines()

for i, line in enumerate(lines):
    if "params_dict[k_norm_key].fill_(1.0)" in line:
        lines[i] = "                        params_dict[k_norm_key].fill_(1.0)"
        break
else:
    raise RuntimeError("fill_ line not found")

path.write_text("\n".join(lines) + "\n")
print("fixed")

In [ ]:
!grep -n -A 3 -B 3 "k_norm_key" /workspace/vllm-gemma4-fix/vllm/model_executor/models/gemma4.py

In [ ]:
# 배포 설정
from huggingface_hub import HfApi, login, whoami

MODEL_ID = "YOUR_HF_USERNAME/gemma4-e4b-civil-complaint-merged"
SERVED_MODEL_NAME = "civil-complaint-sft"
HOST = "0.0.0.0"
PORT = 8000
MAX_MODEL_LEN = 16384
GPU_MEMORY_UTILIZATION = 0.90
LOG_PATH = "vllm_server.log"
login()

if MODEL_ID.startswith("YOUR_HF_USERNAME/"):
    raise ValueError("MODEL_ID를 Hugging Face repo ID로 바꿔주세요.")

In [ ]:
# vLLM OpenAI-compatible server 실행
import subprocess

cmd = [
    "vllm", "serve", MODEL_ID,
    "--served-model-name", SERVED_MODEL_NAME,
    "--host", HOST,
    "--port", str(PORT),
    "--dtype", "bfloat16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--limit-mm-per-prompt", '{"image":0,"audio":0}',
]

log_file = open(LOG_PATH, "w")
server_process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print("server pid:", server_process.pid)
print("log:", LOG_PATH)
print("base url:", f"http://127.0.0.1:{PORT}/v1")

In [ ]:
# 서버 상태 확인
import time
import requests

models_url = f"http://127.0.0.1:{PORT}/v1/models"

for _ in range(60):
    try:
        response = requests.get(models_url, timeout=2)
        if response.ok:
            print(response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print("server not ready. 로그 확인:", LOG_PATH)

In [ ]:
# 간단한 요청 테스트
from openai import OpenAI

client = OpenAI(
    base_url=f"http://127.0.0.1:{PORT}/v1",
    api_key="EMPTY",
)

messages = [
    {"role": "system", "content": "너는 민간 민원 상담 데이터를 처리하는 상담 분석 AI다. 정답만 간결하게 답한다."},
    {"role": "user", "content": "[작업] 분류\n[세부 유형] 상담 내용\n[지시] 이 상담은 일반 문의 상담, 업무 처리 상담, 고충 상담 중 어떤 유형인가?\n\n[상담 내용]\n고객: 예약 가능한 객실과 추가 요금을 알고 싶어요.\n상담사: 예약 가능 여부와 요금을 확인해 드리겠습니다."},
]

result = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages,
    temperature=0,
    max_tokens=128,
)

print(result.choices[0].message.content)